# Matched H&E, mIF and fused feature comparison


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Configuration


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
RUN = PATHS.run
MIF_CASES = RUN / 'cases'
HE_CASES = RUN / 'feature_characterisation' / 'virchow2_he' / 'cases'
LABELS = RUN / 'feature_characterisation' / 'feature_statistics' / 'data' / 'case_level_features_final.parquet'
OUTPUT = RUN / 'feature_characterisation' / 'virchow2_spatial_comparison'
K_MOTIFS = int(os.getenv('KRONOS2_COMPARISON_K', '12'))
PCA_DIM = int(os.getenv('KRONOS2_COMPARISON_PCA_DIM', '32'))
PERMUTATIONS = int(os.getenv('KRONOS2_COMPARISON_PERMUTATIONS', '999'))
SEED = 6211
TISSUE_CONTEXT = False
if not HE_CASES.is_dir():
    raise FileNotFoundError('Run notebook 06 in full mode before this comparison')
OUTPUT.mkdir(parents=True, exist_ok=True)
labels_qc = pd.read_parquet(LABELS, columns=['pcr_label'])
label_counts = labels_qc.pcr_label.value_counts()
if min(label_counts.get(0, 0), label_counts.get(1, 0)) < 2:
    raise ValueError("The report's outcome analyses require at least two labelled cases in each pCR group.")


## Matched features and equal-weight fusion


In [ ]:
meta_columns = ['case_id', 'roi_id', 'patch_id', 'patch_x', 'patch_y', 'patch_stride', 'tumour_fraction', 'tumour_associated_stroma_fraction', 'inflamed_stroma_fraction', 'other_fraction']
case_paths = sorted(MIF_CASES.glob('*/features.parquet'))
assert case_paths
meta_parts, mif_parts, he_parts = ([], [], [])
for path in case_paths:
    case_id = path.parent.name
    he_path = HE_CASES / f'{case_id}.parquet'
    if not he_path.is_file():
        raise FileNotFoundError(f'Missing H&E features for {case_id}. Notebook 06 must finish all cases.')
    mif_all = pd.read_parquet(path)
    he_all = pd.read_parquet(he_path)
    mif_columns = sorted((c for c in mif_all if c.startswith('feature_')))
    he_columns = sorted((c for c in he_all if c.startswith('he_feature_')))
    if len(mif_columns) != 768 or len(he_columns) != 2560:
        raise ValueError(f'Unexpected feature dimensions for {case_id}')
    if mif_all.patch_id.duplicated().any() or he_all.patch_id.duplicated().any():
        raise ValueError(f'Duplicate patch IDs in {case_id}')
    if set(mif_all.patch_id) != set(he_all.patch_id):
        raise ValueError(f'H&E/mIF patch mismatch in {case_id}')
    he_ordered = he_all.set_index('patch_id').loc[mif_all.patch_id, he_columns]
    meta_parts.append(mif_all[meta_columns].copy())
    mif_parts.append(mif_all[mif_columns].to_numpy(dtype=np.float32))
    he_parts.append(he_ordered.to_numpy(dtype=np.float32))
patches = pd.concat(meta_parts, ignore_index=True)
mif_matrix = np.vstack(mif_parts)
he_matrix = np.vstack(he_parts)
assert patches.patch_id.is_unique and np.isfinite(mif_matrix).all() and np.isfinite(he_matrix).all()

def reduced_block(matrix):
    scaled = StandardScaler().fit_transform(matrix).astype(np.float32)
    reduced = PCA(n_components=PCA_DIM, svd_solver='randomized', random_state=SEED).fit_transform(scaled)
    return StandardScaler().fit_transform(reduced).astype(np.float32) / np.sqrt(PCA_DIM)
mif_block = reduced_block(mif_matrix)
he_block = reduced_block(he_matrix)
del mif_matrix, he_matrix, mif_parts, he_parts
context = StandardScaler().fit_transform(patches[['tumour_fraction', 'tumour_associated_stroma_fraction', 'inflamed_stroma_fraction', 'other_fraction']].fillna(0).to_numpy(dtype=np.float32)) / np.sqrt(4)
feature_modes = {'mif_only': mif_block, 'he_only': he_block, 'mif_he_fused': np.column_stack([mif_block, he_block])}
if TISSUE_CONTEXT:
    feature_modes = {name: np.column_stack([matrix, context]) for name, matrix in feature_modes.items()}


## Spatial and case-level tests


In [ ]:
groups = {}
sources, targets = ([], [])
for key, indices in patches.groupby(['case_id', 'roi_id'], sort=False).groups.items():
    idx = np.asarray(list(indices), dtype=np.int32)
    groups[key] = idx
    rows = patches.iloc[idx]
    stride = int(rows.patch_stride.mode().iloc[0])
    lookup = {(int(x), int(y)): int(i) for i, x, y in zip(idx, rows.patch_x, rows.patch_y)}
    for i, x, y in zip(idx, rows.patch_x, rows.patch_y):
        for dx, dy in [(stride, 0), (0, stride), (stride, stride), (stride, -stride)]:
            neighbor = lookup.get((int(x) + dx, int(y) + dy))
            if neighbor is not None:
                sources.append(int(i))
                targets.append(neighbor)
sources = np.asarray(sources, dtype=np.int32)
targets = np.asarray(targets, dtype=np.int32)
if len(sources) == 0:
    raise RuntimeError('No patch-neighbour edges were found')

def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    result = np.full(len(p), np.nan, dtype=float)
    valid = np.isfinite(p)
    order = np.argsort(p[valid])
    ranked = p[valid][order]
    corrected = np.minimum.accumulate((ranked * len(ranked) / np.arange(1, len(ranked) + 1))[::-1])[::-1]
    restored = np.empty(len(ranked), dtype=float)
    restored[order] = np.minimum(corrected, 1)
    result[valid] = restored
    return result

def spatial_test(labels, seed):
    observed = np.bincount(labels[sources][labels[sources] == labels[targets]], minlength=K_MOTIFS)
    null_counts = np.empty((PERMUTATIONS, K_MOTIFS), dtype=np.int32)
    rng = np.random.default_rng(seed)
    for iteration in range(PERMUTATIONS):
        shuffled = labels.copy()
        for idx in groups.values():
            shuffled[idx] = rng.permutation(shuffled[idx])
        same = shuffled[sources] == shuffled[targets]
        null_counts[iteration] = np.bincount(shuffled[sources][same], minlength=K_MOTIFS)
    expected = null_counts.mean(axis=0)
    p = (1 + (null_counts >= observed).sum(axis=0)) / (PERMUTATIONS + 1)
    table = pd.DataFrame({'motif': np.arange(1, K_MOTIFS + 1), 'observed_edges': observed, 'expected_edges': expected, 'fold_enrichment': (observed + 0.5) / (expected + 0.5), 'p_value': p})
    table['fdr_bh'] = bh_adjust(table.p_value)
    return table

def case_summary(labels):
    frame = patches[['case_id', 'roi_id', 'patch_id']].copy()
    frame['motif'] = labels + 1
    proportions = pd.crosstab(frame.case_id, frame.motif, normalize='index').reindex(columns=range(1, K_MOTIFS + 1), fill_value=0)
    proportions.columns = [f'motif_{k:02d}' for k in proportions.columns]
    rows = []
    for case_id, subset in frame.groupby('case_id'):
        idx = subset.index.to_numpy()
        counts = np.bincount(labels[idx], minlength=K_MOTIFS)
        probabilities = counts[counts > 0] / counts.sum()
        entropy = float(stats.entropy(probabilities) / np.log(K_MOTIFS))
        case_mask = patches.iloc[sources].case_id.to_numpy() == case_id
        homotypic = float(np.mean(labels[sources[case_mask]] == labels[targets[case_mask]]))
        rows.append({'case_id': case_id, 'patch_count': len(idx), 'roi_count': subset.roi_id.nunique(), 'motif_entropy': entropy, 'homotypic_edge_fraction': homotypic})
    result = pd.DataFrame(rows).set_index('case_id').join(proportions)
    return (result.reset_index(), frame)

def case_pcr_tests(profiles, labels_frame):
    joined = profiles.merge(labels_frame[['case_id', 'pcr_label']], on='case_id', how='left')
    joined = joined.dropna(subset=['pcr_label'])
    if joined.case_id.duplicated().any():
        raise ValueError('pCR analysis requires one row per case')
    metrics = [f'motif_{k:02d}' for k in range(1, K_MOTIFS + 1)] + ['motif_entropy', 'homotypic_edge_fraction']
    rows = []
    for metric in metrics:
        positive = joined.loc[joined.pcr_label == 1, metric].to_numpy(dtype=float)
        negative = joined.loc[joined.pcr_label == 0, metric].to_numpy(dtype=float)
        if len(positive) < 2 or len(negative) < 2:
            continue
        if np.all(np.concatenate([positive, negative]) == positive[0]):
            p_value, rank_biserial = (1.0, 0.0)
        else:
            test = stats.mannwhitneyu(positive, negative, alternative='two-sided')
            p_value = float(test.pvalue)
            rank_biserial = float(2 * test.statistic / (len(positive) * len(negative)) - 1)
        rows.append({'metric': metric, 'pcr_n': len(positive), 'non_pcr_n': len(negative), 'pcr_median': float(np.median(positive)), 'non_pcr_median': float(np.median(negative)), 'rank_biserial': rank_biserial, 'p_value': p_value})
    table = pd.DataFrame(rows)
    table['fdr_bh'] = bh_adjust(table.p_value)
    return (table.sort_values('p_value'), joined)


## Three-modality comparison


In [ ]:
pcr_labels = pd.read_parquet(LABELS, columns=['case_id', 'pcr_label']).drop_duplicates('case_id')
summary_rows = []
for mode_index, (mode, features) in enumerate(feature_modes.items()):
    mode_output = OUTPUT / mode
    mode_output.mkdir(parents=True, exist_ok=True)
    model = MiniBatchKMeans(n_clusters=K_MOTIFS, batch_size=2048, n_init=5, max_iter=300, random_state=SEED)
    motifs = model.fit_predict(features).astype(np.int32)
    spatial = spatial_test(motifs, SEED + 100 * mode_index)
    profiles, assignments = case_summary(motifs)
    pcr, labelled = case_pcr_tests(profiles, pcr_labels)
    assignments.to_parquet(mode_output / 'patch_motifs.parquet', index=False)
    spatial.to_csv(mode_output / 'motif_spatial_pvalues.csv', index=False)
    profiles.to_csv(mode_output / 'case_profiles.csv', index=False)
    pcr.to_csv(mode_output / 'pcr_case_level_pvalues.csv', index=False)
    ordered_spatial = spatial.sort_values('motif', ascending=False)
    spatial_effect = np.log2(ordered_spatial.fold_enrichment)
    y = np.arange(len(ordered_spatial))
    fig, ax = plt.subplots(figsize=(9, 6.5))
    colours = np.where(spatial_effect >= 0, '#E45756', '#4C78A8')
    ax.hlines(y, 0, spatial_effect, color=colours, linewidth=2)
    ax.scatter(spatial_effect, y, c=colours, s=65, zorder=3)
    ax.axvline(0, color='#333333', linewidth=0.9)
    ax.set_yticks(y, [f'M{k}' for k in ordered_spatial.motif])
    ax.set_xlabel('log2 observed / expected same-motif edges')
    ax.set_title(f'{mode}: motif spatial permutation test')
    ax.margins(x=0.18, y=0.18)
    for yy, effect, p in zip(y, spatial_effect, ordered_spatial.p_value):
        ax.annotate(f'p={p:.3f}', (effect, yy), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
    fig.text(0.5, 0.01, f'{PERMUTATIONS} within-ROI label permutations; raw p-values shown', ha='center', fontsize=9)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(mode_output / 'motif_spatial_pvalues.png', dpi=220)
    plt.close(fig)
    ordered_pcr = pcr.sort_values('rank_biserial')
    effect = ordered_pcr.rank_biserial.to_numpy()
    y = np.arange(len(ordered_pcr))
    fig, ax = plt.subplots(figsize=(9.5, 7))
    colours = np.where(effect >= 0, '#E45756', '#4C78A8')
    ax.hlines(y, 0, effect, color=colours, linewidth=2)
    ax.scatter(effect, y, c=colours, s=65, zorder=3)
    ax.axvline(0, color='#333333', linewidth=0.9)
    ax.set_yticks(y, ordered_pcr.metric.str.replace('_', ' '))
    ax.set_xlabel('Rank-biserial effect; positive is higher in pCR')
    ax.set_title(f'{mode}: exploratory case-level pCR associations')
    ax.margins(x=0.13, y=0.18)
    for yy, row in enumerate(ordered_pcr.itertuples(index=False)):
        ax.annotate(f'p={row.p_value:.3f}', (row.rank_biserial, yy), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
    fig.text(0.5, 0.01, 'Raw p-values shown; adjusted values remain in the CSV table', ha='center', fontsize=9)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(mode_output / 'pcr_effects_raw_p.png', dpi=220)
    plt.close(fig)
    summary_rows.append({'mode': mode, 'cases': int(profiles.case_id.nunique()), 'patches': int(len(assignments)), 'labelled_cases': int(len(labelled)), 'min_raw_p': float(pcr.p_value.min()), 'raw_p_below_0p05': int((pcr.p_value < 0.05).sum()), 'adjusted_p_below_0p05': int((pcr.fdr_bh < 0.05).sum())})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT / 'mode_comparison.csv', index=False)
(OUTPUT / 'manifest.json').write_text(json.dumps({'cases': int(patches.case_id.nunique()), 'patches': int(len(patches)), 'modes': list(feature_modes), 'k_motifs': K_MOTIFS, 'permutations': PERMUTATIONS, 'spatial_smoothing': False, 'tissue_context': TISSUE_CONTEXT, 'interpretation': 'Exploratory comparison. Motif labels are not biologically equivalent across modes. Raw p-values in figures are not multiplicity-adjusted.'}, indent=2), encoding='utf-8')
summary
